# 04 · Inspector-comment text (Phase 2)

**Purpose.** Phase 1 ended with model (b), a 33-feature logistic regression, finding +2.4 more Priority violations per 100 first-half visits than Rule 1 on the clean year 2024, narrowly short of the +3 bar. Tuning and more complex algorithms did not move it: the structured data has run out of information. This notebook asks whether the **free-text comments inspectors wrote at earlier visits** add information model (b) does not have.

**The ladder (PLAN.md, Phase 2), cheapest first.** Each experiment is added *on top of model (b)* and kept only if it beats the current best:

| Step | Text representation | Cost |
|---|---|---|
| E1 | TF-IDF of words and word pairs → logistic regression | seconds |
| E2 | Frozen sentence embeddings (MiniLM) → logistic regression | minutes |
| E3 | Fine-tuned small transformer (DistilBERT) | about 1.3 h on this Mac (§7) |
| E4 | Tuning of the best text model (past visits included, text length, model, learning rate) | only if E3 adds value |

**Guardrails (fixed before running)**
- **Baseline and bar (P1).** Model (b) is reproduced exactly first (§1). Every text model is compared with model (b) *and* with Rule 1; the bar stays +3 hits per 100 visits over Rule 1 with a 95% CI excluding zero.
- **Protocol (P2).** Settings are chosen on folds 1–2 (predicting 2022 and 2023 from earlier years) by mean PR-AUC, as in notebook 03b. 2024 is judged once at the end and never used for selection. No test-period row is built.
- **Keep rule.** An experiment is kept if its best configuration beats the current best by more than **0.002 PR-AUC** on folds 1–2, the noise level used in notebook 03b. **Stop rule:** stop after two experiments in a row add nothing.
- **How text enters (stacking).** Each text model is trained on text alone to predict a hit. Its score becomes **one extra input** to model (b). Scores on training rows are out-of-fold (5 groups by establishment), so model (b) never sees a text score fitted on the same row.
- **Decision time (P4, R15).** A row's text comes only from visits **before the first day of its month**, de-duplicated within the row. Asserted, plus a truncation test.
- **Personal data (P7, R18).** Emails, phone numbers and likely personal names are masked before any model sees the text, and every comment shown in this notebook passes a name scan.

## 0 · Setup

In [1]:
import logging
import os
import re
import time
import warnings
from collections import Counter
from datetime import date

os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'
warnings.filterwarnings('ignore', message='IProgress not found')
logging.getLogger('torch.distributed.elastic.multiprocessing.redirects').setLevel(logging.ERROR)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sentence_transformers
import sklearn
import torch
import transformers
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, brier_score_loss
from sklearn.model_selection import GroupKFold
from sklearn.neighbors import BallTree
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from inspections import config

warnings.filterwarnings('ignore', message='This pattern is interpreted as a regular expression')
transformers.logging.set_verbosity_error()
transformers.logging.disable_progress_bar()
logging.getLogger('huggingface_hub').setLevel(logging.ERROR)
pd.set_option('display.width', 160)
pd.set_option('display.max_colwidth', 120)
EARTH_RADIUS_M = 6_371_000
SELECTION_FOLDS = [2022, 2023]
ALL_FOLDS = [2022, 2023, 2024]
NOISE = 0.002
DEVICE = 'mps' if torch.backends.mps.is_available() else 'cpu'

visits = pd.read_parquet(config.PROCESSED / 'visits.parquet')
violations = pd.read_parquet(config.PROCESSED / 'violations.parquet')
licenses = pd.read_parquet(config.PROCESSED / 'licenses.parquet')
print(f'{len(visits):,} visits · {len(violations):,} violation rows')
print(f'scikit-learn {sklearn.__version__} · torch {torch.__version__} ({DEVICE}) · transformers {transformers.__version__} '
      f'· sentence-transformers {sentence_transformers.__version__}')

193,852 visits · 835,847 violation rows
scikit-learn 1.9.1 · torch 2.15.0 (mps) · transformers 5.22.2 · sentence-transformers 6.2.0


**Carried over from notebook 03b, unchanged:** the metric helpers, the `check()` helper, the violation-type groups and the feature builder at model (b)'s settings (α = 0.2, last N = 2, local rates within 1 km over 90 + 180 days). Only the parts model (b) uses are kept: the history, trend and area groups. §1 proves the copy reproduces model (b) exactly.

In [2]:
def per_month(df, score, q=0.5):
    rows = []
    for month, g in df.groupby('month', sort=True):
        k = int(len(g) * q)
        top = g.sort_values([score, 'tiebreak'], ascending=[False, True]).head(k)
        rows.append((month, top.hit.sum(), k, g.hit.sum(), len(g)))
    return pd.DataFrame(rows, columns=['month', 'hits', 'visits', 'month_hits', 'month_visits']).set_index('month')

def hit_rate(pm):
    return pm.hits.sum() / pm.visits.sum()

def bootstrap_diff(pm_a, pm_b, n=2000, seed=config.SEED):
    r = np.random.default_rng(seed)
    idx = r.integers(0, len(pm_a), size=(n, len(pm_a)))
    ha, va, hb, vb = pm_a.hits.to_numpy(), pm_a.visits.to_numpy(), pm_b.hits.to_numpy(), pm_b.visits.to_numpy()
    return np.percentile(ha[idx].sum(1) / va[idx].sum(1) - hb[idx].sum(1) / vb[idx].sum(1), [2.5, 97.5])

def days_earlier(df, score):
    shifts = []
    for _, g in df.groupby('month'):
        ts = g.visit_ts.dt.tz_localize(None)
        slots = np.sort(ts.to_numpy())
        new_date = pd.Series(slots, index=g.sort_values([score, 'tiebreak'], ascending=[False, True]).index)
        hits = g.index[g.hit]
        shifts.append(((ts.loc[hits] - new_date.loc[hits]).dt.total_seconds() / 86400).to_numpy())
    return np.concatenate(shifts).mean()

CHECKS = []
def check(risk, what, ok, detail=''):
    CHECKS.append({'risk': risk, 'check': what, 'result': 'pass' if ok else 'FAIL', 'detail': detail})
    assert ok, f'{risk} failed: {what} ({detail})'
    print(f'✓ {risk}: {what}' + (f'  [{detail}]' if detail else ''))

In [3]:
VIOLATION_GROUPS = {   # first matching group wins, so order matters
    'pests': r'pest|insect|rodent|animal|vermin',
    'temperature': r'temperature|holding|cooling|cooked|cooking|reheat|thaw|thermometer|time as a public health',
    'handwashing': r'hand|when to wash|hygien|hair|eating, drinking|tobacco|clean cloth|outer cloth|clothing|glove'
                   r'|location  accessible|number  convenient',
    'contamination': r'contamina|separation|segregat|food protection|raw|adulterat|unsafe|discard|storage|stored'
                     r'|container|label|date mark|wrapped|packag|ice|approved source|sources|shell ?stock|tags & records'
                     r'|consumer advis|allergen|toxic|poisonous|chemical|food preparation',
    'sanitizing': r'food contact|food-contact|sanit|warewash|dishwash|test kit|utensil|wiping|sponge|equipment|drain ?board',
    'management': r'pic|person-in-charge|person in charge|knowledge|certified|manager|employee health|illness|vomit'
                  r'|demonstration|duties|compliance|permit|license|posting|variance|haccp|plan|conformance|procedure'
                  r'|assignment|ceasing operations|anti-choking|reporting',
    'facility': r'floor|wall|ceiling|premises|repair|ventilation|light|fixture|shield|plumb|sewage|toilet|garbage'
                r'|refuse|dumpster|receptacle|grease|backflow|water|sink|surface|installed|designed|constructed'
                r'|nonfood|non-food|outside|outer opening|maintained|maintenance|linen|mop|dressing room|locker'
                r'|cleanab|cleaning',
}

def violation_group(description):
    d = description.lower()
    return next((g for g, pattern in VIOLATION_GROUPS.items() if re.search(pattern, d)), 'other')

group_of = {d: violation_group(d) for d in violations.violdesc.dropna().unique()}
violations['group'] = violations.violdesc.map(group_of).fillna('other')
viol_groups = violations.pivot_table(index=['licenseno', 'resultdttm'], columns='group', values='violation',
                                     aggfunc='size', fill_value=0).add_prefix('n_')
GROUP_COLS = list(viol_groups.columns)

In [4]:
ESCALATIONS = ['HE_FailExt', 'HE_Hearing', 'HE_TSOP', 'HE_Closure']

def asof(keys, table, time_col, by):
    """For each key row (by, t0): the latest `table` row strictly before t0."""
    left = keys.reset_index().sort_values('t0')
    out = pd.merge_asof(left, table.sort_values(time_col), left_on='t0', right_on=time_col, by=by,
                        allow_exact_matches=False)
    return out.set_index('index').sort_index()


def history_tables(vs, ewm_alpha=0.2, last_n=2):
    """Running per-license history: one table over all visits, one over routine (cycle-opening) visits."""
    v = vs.sort_values(['licenseno', 'visit_ts']).reset_index(drop=True)
    v['ts'] = v.visit_ts.dt.tz_localize(None)
    v['is_fail'] = v.role.eq('fail')
    g = v.groupby('licenseno')
    v['cum_visits'] = g.cumcount() + 1
    v['cum_escalations'] = v.result.isin(ESCALATIONS).groupby(v.licenseno).cumsum()
    v['first_ts'] = g.ts.transform('min')
    v['last_fail_ts'] = v.ts.where(v.is_fail).groupby(v.licenseno).ffill()

    o = v[v.opener].join(viol_groups, on=['licenseno', 'resultdttm'])
    o[GROUP_COLS] = o[GROUP_COLS].fillna(0)
    o['severe'] = o.n_severe.gt(0)
    o['old_code'] = o.ts.dt.year < 2020
    go = o.groupby('licenseno')
    o['cum_routine'] = go.cumcount() + 1
    o['cum_routine_fail'] = go.is_fail.cumsum()
    o['cum_routine_severe'] = go.severe.cumsum()
    o['prev2_fail'] = go.is_fail.shift()
    o['prev2_severe'] = go.severe.shift()
    for c in ['severe', 'is_fail', 'n_viol', 'n_temperature', 'n_contamination', 'n_pests']:
        o[f'last3_{c}'] = go[c].transform(lambda s: s.astype(float).rolling(last_n, min_periods=1).sum())
    o['ewm_severe'] = go.severe.transform(lambda s: s.astype(float).ewm(alpha=ewm_alpha).mean())
    o['ewm_fail'] = go.is_fail.transform(lambda s: s.astype(float).ewm(alpha=ewm_alpha).mean())
    o['new_code_routine'] = (~o.old_code).groupby(o.licenseno).cumsum()
    o['new_code_severe'] = (o.severe & ~o.old_code).groupby(o.licenseno).cumsum()
    return v, o


def history_features(keys, v, o):
    """Model (b)'s history and trend groups. Also returns the as-of routine-visit table (used for the violation-mix control)."""
    lv = asof(keys, v[['licenseno', 'ts', 'role', 'cum_visits', 'cum_escalations', 'first_ts', 'last_fail_ts']],
              'ts', 'licenseno')
    rcols = ['licenseno', 'ts', 'is_fail', 'severe', 'old_code', 'n_viol', 'n_severe', 'cum_visits', 'cum_routine',
             'cum_routine_fail', 'cum_routine_severe', 'prev2_fail', 'prev2_severe', 'ewm_severe', 'ewm_fail',
             'new_code_routine', 'new_code_severe'] + GROUP_COLS + [c for c in o.columns if c.startswith('last3_')]
    lr = asof(keys, o[rcols].rename(columns={'ts': 'routine_ts', 'cum_visits': 'cum_visits_at_routine'}),
              'routine_ts', 'licenseno')
    F = pd.DataFrame(index=keys.index)
    # history
    F['has_history'] = lv.ts.notna()
    F['prev_fail'] = lr.is_fail.astype(float)
    F['prev_severe'] = lr.severe.astype(float)
    F['prev_n_viol'] = lr.n_viol
    F['prev_n_severe'] = lr.n_severe
    F['prev_old_code'] = lr.old_code.astype(float)
    F['prev2_fail'] = lr.prev2_fail.astype(float)
    F['prev2_severe'] = lr.prev2_severe.astype(float)
    F['n_prior_routine'] = lr.cum_routine.fillna(0)
    F['prior_fail_rate'] = lr.cum_routine_fail / lr.cum_routine
    F['prior_severe_rate'] = lr.cum_routine_severe / lr.cum_routine
    F['followups_last_cycle'] = lv.cum_visits - lr.cum_visits_at_routine
    F['ever_escalated'] = lv.cum_escalations.gt(0).astype(float).where(lv.ts.notna())
    F['last_visit_filed'] = lv.role.eq('filed').astype(float).where(lv.ts.notna())
    F['days_since_last_visit'] = (keys.t0 - lv.ts).dt.days
    F['days_since_last_routine'] = (keys.t0 - lr.routine_ts).dt.days
    F['days_since_last_fail'] = (keys.t0 - lv.last_fail_ts).dt.days
    F['days_since_first_visit'] = (keys.t0 - lv.first_ts).dt.days
    # recent trend
    for c in [c for c in lr.columns if c.startswith('last3_')]:
        F[c] = lr[c]
    F['ewm_severe'] = lr.ewm_severe
    F['ewm_fail'] = lr.ewm_fail
    F['new_code_severe_rate'] = lr.new_code_severe / lr.new_code_routine.replace(0, np.nan)
    return F, lr


def area_features(keys, o, radius_m=1000, windows=(90, 180)):
    """Recent outcomes of routine visits to *any* establishment within 1 km: a proxy for local inspector strictness."""
    ref = o[['licenseno', 'ts', 'severe', 'is_fail']].join(licenses[['lat', 'lon']], on='licenseno').dropna(subset=['lat'])
    ref = ref[ref.ts >= keys.t0.min() - pd.Timedelta(days=max(windows))]   # older visits can never fall in a window
    xy = keys[['licenseno']].join(licenses[['lat', 'lon']], on='licenseno')
    ok = xy.lat.notna().to_numpy()
    tree = BallTree(np.radians(ref[['lat', 'lon']].to_numpy()), metric='haversine')
    near = tree.query_radius(np.radians(xy.loc[ok, ['lat', 'lon']].to_numpy()), r=radius_m / EARTH_RADIUS_M)
    rt, rs, rf = ref.ts.to_numpy(), ref.severe.to_numpy(), ref.is_fail.to_numpy()
    t0 = keys.t0.to_numpy()[ok]
    F = pd.DataFrame(index=keys.index)
    for days in windows:
        lo = t0 - np.timedelta64(days, 'D')
        sev, fail, n = (np.full(ok.sum(), np.nan) for _ in range(3))
        for k, i in enumerate(near):
            m = (rt[i] < t0[k]) & (rt[i] >= lo[k])
            n[k] = m.sum()
            if n[k]:
                sev[k], fail[k] = rs[i][m].mean(), rf[i][m].mean()
        for name, arr in [('severe_rate', sev), ('fail_rate', fail), ('routine_visits', n)]:
            col = np.full(len(keys), np.nan)
            col[ok] = arr
            F[f'area_{name}_{days}d'] = col
    return F


def build_model_b(keys, visits_src):
    """Model (b)'s 33 features as of each row's t0, plus the as-of routine-visit table."""
    v, o = history_tables(visits_src)
    hist, lr = history_features(keys, v, o)
    return pd.concat([hist, area_features(keys, o)], axis=1).astype(float), lr

## 1 · Reproduce model (b) first

Rows: the 2020–2024 prediction set (train + validation); the test period is not built. Model (b) must reproduce notebook 03b's clean-year result **exactly** before anything is added to it.

In [5]:
rows = visits[visits.in_prediction_set & visits.split.isin(['train', 'valid'])].copy()
check('R14', 'no test-period rows built', rows.split.isin(['train', 'valid']).all(), f'{len(rows):,} rows')
y = rows.hit.to_numpy()
year = rows.visit_ts.dt.year.to_numpy()
lic = rows.licenseno.to_numpy()
keys = rows[['licenseno']].copy()
keys['t0'] = rows.visit_ts.dt.tz_localize(None).dt.to_period('M').dt.start_time

X_b, lr_asof = build_model_b(keys, visits)
check('P1', 'model (b) has its 33 features', X_b.shape[1] == 33, f'{X_b.shape[0]:,} rows × {X_b.shape[1]} features')

in_eval = np.isin(year, ALL_FOLDS)
ev = rows.loc[in_eval, ['month', 'visit_ts', 'hit', 'category']].copy()
ev['tiebreak'] = np.random.default_rng(config.SEED).random(len(ev))
ev['actual'] = -ev.visit_ts.dt.tz_localize(None).astype('int64')
ev['oracle'] = ev.hit.astype(int)
Xe = X_b.loc[ev.index]
ev['rule1'] = 2 * Xe.prev_severe.fillna(0) + Xe.prev_fail.fillna(0) + Xe.days_since_last_visit.fillna(9999) / 1e5

✓ R14: no test-period rows built  [27,457 rows]


✓ P1: model (b) has its 33 features  [27,457 rows × 33 features]


In [6]:
def make_logreg(C=0.003):
    """Model (b)'s classifier: L2 logistic regression, C = 0.003 (notebook 03b)."""
    return make_pipeline(SimpleImputer(strategy='median', add_indicator=True), StandardScaler(),
                         LogisticRegression(C=C, l1_ratio=0, solver='lbfgs', max_iter=5000, random_state=config.SEED))

def backtest(X, folds):
    """Out-of-sample probabilities for each fold year, training on 2020 .. year-1."""
    oof = pd.Series(np.nan, index=rows.index)
    for Y in folds:
        tr, te = year < Y, year == Y
        oof[te] = make_logreg().fit(X[tr], y[tr]).predict_proba(X[te])[:, 1]
    return oof

oof_b = backtest(X_b, ALL_FOLDS)
ev['model_b'] = oof_b.loc[ev.index].to_numpy()
hr_b = hit_rate(per_month(ev[ev.visit_ts.dt.year == 2024], 'model_b'))
logged = pd.read_csv(config.RESULTS / 'experiments.csv')
hr_03b = logged.loc[(logged.notebook == '03b_tuning_round') & (logged.experiment == 'Tuned logistic regression (03b)'), 'HR@50'].item()
check('R23', 'model (b) reproduced exactly (2024 first-half hit rate)', abs(hr_b - hr_03b) < 1e-12,
      f'{hr_b:.4f} here vs {hr_03b:.4f} in notebook 03b')

✓ R23: model (b) reproduced exactly (2024 first-half hit rate)  [0.3480 here vs 0.3480 in notebook 03b]


## 2 · The comment text

Every violation row carries the inspector's free-text comment (`comments`) next to the standard Food Code description (`violdesc`). The description says *which* rule was broken; the comment says what the inspector saw. This section audits the comments, cleans them, and builds the text each row is allowed to see.

### 2.1 · Masking personal data (P7, R18)

The comments contain contact details and a few personal names: the health department's phone numbers and staff email addresses (some named, e.g. an inspector's own), and occasionally the person in charge by name ("PIC *first last*", "Mr. *surname*"). These identify people, not kitchens; an inspector's email would also let a model learn *who inspected* rather than *what was found* (R17). They are removed before any model sees the text. Food Code tags such as `(P)` are stripped too (P6). Only counts are printed here, never the matches.

In [7]:
EMAIL = re.compile(r'[\w.\-]+@[\w\-]+(\.[\w\-]+)+', re.I)
PHONE = re.compile(r'\(?\b\d{3}\)?[-. ]?\d{3}[-. ]\d{4}\b')
TAG = re.compile(r'\((P|Pf|PF|C)\)')
TITLED = re.compile(r'\b(Mr|Mrs|Ms|Miss)\.? [A-Z][a-z]+( [A-Z][a-z]+)?')
ROLE_NAME = re.compile(r'\b(PIC|P\.I\.C\.|[Mm]anager|[Oo]wner|[Cc]hef|[Ss]upervisor)([,:]| named| is)? ([A-Z][a-z]+)( [A-Z][a-z]+)?')

def clean_comment(s):
    s = re.sub(r'\s+', ' ', s)                          # first, so a double space cannot hide a phone number
    s = EMAIL.sub(' ', s)
    s = PHONE.sub(' ', s)
    s = TAG.sub(' ', s)
    s = TITLED.sub(' ', s)
    s = ROLE_NAME.sub(lambda m: m.group(1) + ' ', s)     # keep the role word, drop what may be a name
    return re.sub(r'\s+', ' ', s).strip()

com = violations.loc[violations.comments.notna(), ['licenseno', 'resultdttm', 'visit_ts', 'comments', 'violdesc']].copy()
raw = com.comments
masked = pd.Series({'email address': raw.str.contains(EMAIL).sum(), 'phone number': raw.str.contains(PHONE).sum(),
                    'Food Code tag': raw.str.contains(TAG).sum(), 'title + name ("Mr. …")': raw.str.contains(TITLED).sum(),
                    'role + capitalised word ("PIC …")': raw.str.contains(ROLE_NAME).sum()}, name='comments affected')
com['text'] = raw.map(clean_comment)
com = com[com.text.str.len() > 0]
com['ts'] = com.visit_ts.dt.tz_localize(None)
check('R18', 'no email addresses or phone numbers left in the cleaned text',
      not com.text.str.contains(EMAIL).any() and not com.text.str.contains(PHONE).any(), f'{len(com):,} comments')
masked.to_frame()

✓ R18: no email addresses or phone numbers left in the cleaned text  [812,396 comments]


,comments affected
email address,2173
phone number,313
Food Code tag,84
"title + name (""Mr. …"")",15
"role + capitalised word (""PIC …"")",1419


The role-plus-capitalised-word rule is deliberately broad: it also removes harmless words ("Manager Certification"), which costs nothing for modelling. Names written without a role word can remain in the text the models read. That is acceptable for modelling, which runs locally, but not for anything *displayed*. Every snippet shown in this notebook therefore goes through a second, stricter filter (§9).

### 2.2 · What the comments look like over time (R17)

In [8]:
yr = com.visit_ts.dt.year
by_year = pd.DataFrame({
    'violation rows with a comment': violations.groupby(violations.visit_ts.dt.year).comments.apply(lambda s: s.notna().mean()),
    'median length (characters)': com.text.str.len().groupby(yr).median(),
    'Food Code tag in the description': violations.groupby(violations.visit_ts.dt.year).violdesc.apply(
        lambda s: s.str.contains(TAG, na=False).mean()),
}).loc[2008:]
by_year.style.format({'violation rows with a comment': '{:.0%}', 'median length (characters)': '{:.0f}',
                      'Food Code tag in the description': '{:.0%}'})

,violation rows with a comment,median length (characters),Food Code tag in the description
visit_ts,,,
2008,96%,57,0%
2009,95%,57,0%
2010,96%,63,0%
2011,96%,64,0%
2012,96%,59,0%
2013,98%,63,0%
2014,98%,65,0%
2015,99%,74,0%
2016,99%,75,0%


- **Coverage is complete:** 95–100% of violation rows carry a comment in every year.
- **Comments get longer:** the median rises from about 60 characters (2008–2012) to about 95 (2021–2024) and over 100 since. Writing style drifts. This is one reason text models are checked year by year (§8).
- **Food Code tags live in the description, not the comment.** The `(P)`, `(Pf)`, `(C)` tags appear only in `violdesc` from 2019 on. The comments carry no category labels, so a text model cannot read the answer off a tag.

### 2.3 · Repeats and edits (R15, R4)

**Repeats.** When a kitchen passes its follow-up, the pass visit lists the earlier violations again with the same comments. A comment repeated at a later visit is not new evidence, so text is **de-duplicated within each row** (§2.4).

**Edits after the fact.** R4 found that status fields are updated after follow-ups. If comments were also edited later, an old comment could carry future information. One visible trace would be comments that mention a date well after their visit.

In [9]:
role = com.join(visits.set_index(['licenseno', 'resultdttm']).role, on=['licenseno', 'resultdttm']).role
dup = com.duplicated(['licenseno', 'text'])
print('share of comments that repeat an earlier comment of the same establishment, by visit result:')
print(dup.groupby(role).mean().map('{:.0%}'.format).to_string())

DATE = re.compile(r'\b(\d{1,2})/(\d{1,2})/(\d{2}|\d{4})\b')
d = com.text.str.extract(DATE).dropna().astype(int)
d_year = d[2].where(d[2] > 100, d[2] + 2000)
mentioned = pd.to_datetime(pd.DataFrame({'year': d_year, 'month': d[0].clip(1, 12), 'day': d[1].clip(1, 28)}), errors='coerce')
ahead = (mentioned - com.loc[d.index, 'ts']).dt.days
print(f'\ncomments mentioning a date: {len(d):,} ({len(d) / len(com):.1%}); '
      f'of these, a date > 14 days after the visit: {(ahead > 14).sum()} ({(ahead > 14).mean():.1%})')
check('R4', 'comments carry almost no dates from after their visit (< 0.1% of all comments)', (ahead > 14).sum() / len(com) < 0.001,
      f'{(ahead > 14).sum()} of {len(com):,}')

share of comments that repeat an earlier comment of the same establishment, by visit result:
role
fail     21%
filed    64%
pass     98%



comments mentioning a date: 3,942 (0.5%); of these, a date > 14 days after the visit: 47 (1.2%)
✓ R4: comments carry almost no dates from after their visit (< 0.1% of all comments)  [47 of 812,396]


- **Pass visits repeat almost everything** (about 98%), as expected; fails mostly carry new text.
- **Little sign of later edits:** under 0.5% of comments mention any date. Of those, about 1% mention a date more than two weeks after the visit, and a manual read shows these are mostly deadlines and expiry dates ("permit expired 12/31", "provide report by …"), not reports of later events. A snapshot cannot rule out silent edits, but nothing points to them.

### 2.4 · The text each row may see (P4, P5)

For a row with decision date t0 (the first day of its month): the comments from the establishment's **last *n* routine cycles before t0**, i.e. its last *n* routine visits and every follow-up visit after them, all before t0. Comments are ordered **newest visit first** and de-duplicated, so a comment repeated at a pass visit appears once. *n* = 1, 2 and 3 are built; the choice is made on folds 1–2 in E1 (P5).

Rows with no earlier routine visit, or with earlier visits that cited nothing, get empty text. The text model then outputs its base rate, and model (b) already knows these rows have no record.

In [10]:
def text_windows(keys, visits_src, com_src, n_cycles):
    """For each row (licenseno, t0): de-duplicated comments from the last n routine cycles before t0, newest first."""
    op = visits_src.loc[visits_src.opener, ['licenseno', 'visit_ts']]
    op_by = {k: np.sort(g.visit_ts.dt.tz_localize(None).to_numpy()) for k, g in op.groupby('licenseno')}
    cs = com_src.sort_values(['licenseno', 'ts'], kind='stable')
    c_by = {k: (g.ts.to_numpy(), g.text.to_numpy()) for k, g in cs.groupby('licenseno')}
    texts, latest = [], []
    for license_no, t0 in zip(keys.licenseno.to_numpy(), keys.t0.to_numpy()):
        ots, cc = op_by.get(license_no), c_by.get(license_no)
        k = 0 if ots is None else np.searchsorted(ots, t0, side='left')     # routine visits strictly before t0
        if cc is None or k == 0:
            texts.append([]); latest.append(np.datetime64('NaT', 'us')); continue
        ts, tx = cc
        lo, hi = np.searchsorted(ts, ots[max(k - n_cycles, 0)], side='left'), np.searchsorted(ts, t0, side='left')
        texts.append(list(dict.fromkeys(tx[lo:hi][::-1])))                  # newest first, de-duplicated
        latest.append(ts[hi - 1] if hi > lo else np.datetime64('NaT', 'us'))
    return pd.DataFrame({'comments': texts, 'latest_ts': latest}, index=keys.index)

WINDOWS = {n: text_windows(keys, visits, com, n) for n in [1, 2, 3]}
TEXT = {n: W.comments.map(' '.join) for n, W in WINDOWS.items()}

for n, W in WINDOWS.items():
    has = W.latest_ts.notna()
    check('R15', f'{n}-cycle text comes only from visits before t0', (W.latest_ts[has] < keys.t0[has]).all(),
          f'{has.mean():.0%} of rows have text')

✓ R15: 1-cycle text comes only from visits before t0  [65% of rows have text]
✓ R15: 2-cycle text comes only from visits before t0  [77% of rows have text]
✓ R15: 3-cycle text comes only from visits before t0  [81% of rows have text]


**Truncation test.** Rebuild the 2-cycle text after deleting every visit and comment from 1 July 2023 onward. Every row decided before that date must get exactly the same text.

In [11]:
CUT = pd.Timestamp('2023-07-01')
early = keys.t0 < CUT
W_cut = text_windows(keys[early], visits[visits.visit_ts.dt.tz_localize(None) < CUT], com[com.ts < CUT], 2)
same = (W_cut.comments.map(' '.join) == TEXT[2][early])
check('R15', 'text unchanged when all later data is deleted (truncation test)', same.all(),
      f'{early.sum():,} rows, {(~same).sum()} mismatches')

✓ R15: text unchanged when all later data is deleted (truncation test)  [19,263 rows, 0 mismatches]


In [12]:
tok = transformers.AutoTokenizer.from_pretrained('distilbert/distilbert-base-uncased')
lengths = {}
for n in [1, 2, 3]:
    t = TEXT[n][TEXT[n].str.len() > 0]
    lengths[f'{n} cycle' + ('s' if n > 1 else '')] = pd.Series([len(ids) for ids in tok(t.tolist(), truncation=False)['input_ids']])
summary = pd.DataFrame({k: {'rows with text': len(v) / len(rows), 'median tokens': v.median(), '90th percentile': v.quantile(.9),
                            'over 256 tokens': (v > 256).mean(), 'over 512 tokens': (v > 512).mean()} for k, v in lengths.items()}).T
summary.style.format({'rows with text': '{:.0%}', 'median tokens': '{:.0f}', '90th percentile': '{:.0f}',
                      'over 256 tokens': '{:.0%}', 'over 512 tokens': '{:.1%}'})

,rows with text,median tokens,90th percentile,over 256 tokens,over 512 tokens
1 cycle,65%,77,238,8%,1.0%
2 cycles,77%,125,399,23%,5.4%
3 cycles,81%,174,555,35%,11.9%


**Truncation (R16, P5).** Comments are short: the median row with text has under 100 tokens even with two cycles. TF-IDF and the embeddings (one comment at a time, §6) see all of the text. Only the fine-tuned transformer (§7) truncates. It reads 256 tokens, which covers the whole text for most rows, and because comments are ordered newest first, truncation drops the *oldest* comments.

## 3 · Pre-check: is there signal, and is it new? (P3)

The expected signal: *among places with a clean last inspection, those with more inspector notes are far riskier, and mentions of mice or "repeat" violations signal higher risk.* This section measures that, then asks the question that matters: **is that signal already in model (b)?**

In [13]:
pre = pd.DataFrame({'hit': y, 'no_priority_last_time': X_b.prev_severe.fillna(0).eq(0),
                    'clean_last_time': X_b.prev_severe.fillna(0).eq(0) & X_b.prev_fail.fillna(0).eq(0),
                    'n_comments': WINDOWS[1].comments.map(len), 'text': TEXT[2].str.lower()}, index=rows.index)
bins, labels = [-1, 0, 2, 5, 100], ['none', '1–2', '3–5', '6 or more']
out = {}
for col, name in [('no_priority_last_time', 'no Priority violation at the last routine visit'),
                  ('clean_last_time', 'clean last routine visit (no Priority violation, not failed)')]:
    sub = pre[pre[col]]
    g = sub.groupby(pd.cut(sub.n_comments, bins, labels=labels), observed=True).hit
    out[(name, 'hit rate')] = g.mean()
    out[(name, 'visits')] = g.size()
pd.DataFrame(out).rename_axis('comments written in the last cycle').style.format(
    lambda v: f'{v:.0%}' if v < 1 else f'{v:,.0f}')

In [14]:
KEYWORDS = {'mice / rodents / droppings': r'\bmice\b|\bmouse\b|rodent|dropping', 'cockroaches': r'cockroach|\broach',
            '"repeat"': r'\brepeat', 'discontinue': r'discontinue', 'person in charge ("PIC")': r'\bpic\b'}
kw = {}
for name, pattern in KEYWORDS.items():
    m = pre.text.str.contains(pattern)
    kw[name] = {'rows mentioning it': m.mean(), 'hit rate if mentioned': pre.hit[m].mean(), 'hit rate if not': pre.hit[~m].mean()}
pd.DataFrame(kw).T.style.format('{:.1%}')

,rows mentioning it,hit rate if mentioned,hit rate if not
mice / rodents / droppings,10.5%,37.5%,19.2%
cockroaches,2.3%,39.1%,20.7%
"""repeat""",1.0%,48.7%,20.9%
discontinue,18.8%,38.9%,17.0%
"person in charge (""PIC"")",25.0%,33.2%,17.1%


**The raw associations hold.** Among visits without a Priority violation last time, the hit rate rises from 10% (no comments in the last cycle) to 32% (six or more). Rows whose last two cycles mention mice or rodents, cockroaches or "repeat" hit 38–49% of the time, against 19–21% for rows without them.

**But raw lift is not the question.** More comments means more violations, and model (b) already counts violations. The test that matters: among visits model (b) rates *equally* risky, do the keywords still separate hits from non-hits? Out-of-fold model (b) scores for 2022–24 answer it.

In [15]:
cond = pre.loc[in_eval].assign(p=oof_b.loc[ev.index].to_numpy())
cond['score band'] = pd.qcut(cond.p, 5, labels=['lowest fifth', '2nd', '3rd', '4th', 'highest fifth'])
res = {}
for name in ['mice / rodents / droppings', 'cockroaches', '"repeat"']:
    m = cond.text.str.contains(KEYWORDS[name])
    res[name] = {'rows (2022–24)': m.sum(), 'observed hit rate': cond.hit[m].mean(), 'model (b) expected': cond.p[m].mean(),
                 'observed − expected (points)': 100 * (cond.hit[m].mean() - cond.p[m].mean())}
pd.DataFrame(res).T.style.format({'rows (2022–24)': '{:,.0f}', 'observed hit rate': '{:.1%}', 'model (b) expected': '{:.1%}',
                                  'observed − expected (points)': '{:+.1f}'})

,rows (2022–24),observed hit rate,model (b) expected,observed − expected (points)
mice / rodents / droppings,"1,954",36.8%,35.2%,+1.6
cockroaches,419,40.6%,36.8%,+3.8
"""repeat""",186,49.5%,45.7%,+3.7


**Reading the pre-check.** Model (b) already expects most of the higher risk. Rows mentioning rodents hit 36.8% of the time against 35.2% expected (+1.6 points). For cockroaches and "repeat" the gap is about +4 points, but on only 419 and 186 rows over three years. Of the raw lifts (+18 to +28 points over rows without the word), **most is already priced in** by the violation counts and history in model (b). What remains is small and spread thin. Any value in the text has to come from many weak cues rather than a few strong words, and that is what the models below test.

## 4 · How a text model is added to model (b)

**Stacking, in three steps per fold year Y** (training years 2020 … Y−1):
1. **Text score for training rows, out-of-fold:** the training rows are split into 5 groups *by establishment*, and each group is scored by a text model fit on the other four. No row's score comes from a model that saw that row, or that establishment.
2. **Text score for year-Y rows:** a text model fit on all training rows.
3. **Model (b) + text score:** the logit of the text score is added as a 34th input to model (b) (same classifier, C = 0.003), fit on the training rows and scored on year Y.

**Why stacking rather than putting thousands of word features straight into model (b):** model (b)'s strong regularisation was tuned for 33 dense inputs. Merging in 28,000 sparse word features would force one penalty on both. Stacking keeps model (b) intact and gives the text one clearly measurable channel. It is also exactly how the fine-tuned transformer must enter (§7), so all experiments are compared on equal terms.

In [16]:
def text_scores(inputs, make_model, folds, inner=5):
    """Per fold year Y: out-of-fold text scores on training rows (GroupKFold by establishment) and scores on year-Y rows."""
    scores = {}
    for Y in folds:
        tr, te = np.where(year < Y)[0], np.where(year == Y)[0]
        s = pd.Series(np.nan, index=rows.index)
        for a, b in GroupKFold(inner).split(tr, groups=lic[tr]):
            s.iloc[tr[b]] = make_model().fit(inputs.iloc[tr[a]], y[tr[a]]).predict_proba(inputs.iloc[tr[b]])[:, 1]
        s.iloc[te] = make_model().fit(inputs.iloc[tr], y[tr]).predict_proba(inputs.iloc[te])[:, 1]
        scores[Y] = s
    return scores

def logit(p):
    p = np.clip(np.asarray(p, float), 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))

def stacked_backtest(score_sets, folds, X=None):
    """Model (b) plus one logit text score per score set; out-of-sample probabilities for each fold year."""
    X = X_b if X is None else X
    oof = pd.Series(np.nan, index=rows.index)
    for Y in folds:
        tr, te = year < Y, year == Y
        Xs = X.assign(**{f'text_{i}': logit(s[Y]) for i, s in enumerate(score_sets)})
        oof[te] = make_logreg().fit(Xs[tr], y[tr]).predict_proba(Xs[te])[:, 1]
    return oof

def text_only(scores, folds):
    """The text score on its own, for each fold year's rows."""
    out = pd.Series(np.nan, index=rows.index)
    for Y in folds:
        out[year == Y] = scores[Y][year == Y]
    return out

def selection_score(oof):
    """Mean PR-AUC over folds 1-2, plus the pooled first-half hit rate on those folds."""
    aps = [average_precision_score(y[year == Y], oof[year == Y]) for Y in SELECTION_FOLDS]
    e = ev[ev.visit_ts.dt.year.isin(SELECTION_FOLDS)].copy()
    e['score'] = oof.loc[e.index].to_numpy()
    return np.mean(aps), hit_rate(per_month(e, 'score'))

TRIALS = []
def record(step, setting, oof, text_alone=None):
    ap, hr = selection_score(oof)
    row = {'step': step, 'setting': setting, 'PR-AUC (folds 1–2)': ap, 'HR@50 (folds 1–2)': hr}
    if text_alone is not None:
        row['text alone: PR-AUC'] = selection_score(text_alone)[0]
    TRIALS.append(row)
    return ap

def show(step):
    t = pd.DataFrame([r for r in TRIALS if r['step'] == step]).drop(columns='step').dropna(axis=1, how='all')
    best = t['PR-AUC (folds 1–2)'].idxmax()
    return (t.style.format({c: '{:.4f}' for c in t.columns if 'PR-AUC' in c} | {'HR@50 (folds 1–2)': '{:.1%}'}, na_rep='')
             .apply(lambda r: ['font-weight: bold' if r.name == best else '' for _ in r], axis=1))

base_ap = record('baseline', 'model (b), no text', oof_b)
SCORES = {}                                        # text score sets on the selection folds, by experiment label
BEST = {'name': 'model (b)', 'ap': base_ap, 'parts': []}
print(f'current best on folds 1–2: model (b) · PR-AUC {base_ap:.4f} · HR@50 {selection_score(oof_b)[1]:.1%}')

def decide(step, candidates):
    """Keep rule: the step's best configuration must beat the current best by more than NOISE (PR-AUC, folds 1-2).
    candidates: (name, PR-AUC, list of score-set labels it stacks)."""
    name, ap, parts = max(candidates, key=lambda c: c[1])
    gain = ap - BEST['ap']
    kept = gain > NOISE
    print(f'{step}: best configuration "{name}" · PR-AUC {ap:.4f} vs current best ({BEST["name"]}) {BEST["ap"]:.4f} '
          f'→ {gain:+.4f} · {"KEPT: adds value" if kept else "adds nothing (within noise)"}')
    if kept:
        BEST.update(name=name, ap=ap, parts=parts)
    return kept

current best on folds 1–2: model (b) · PR-AUC 0.4761 · HR@50 32.9%


## 5 · E1 · TF-IDF + logistic regression

Words and word pairs, weighted by TF-IDF (`min_df` = 10 rows, `max_df` = 50%, sublinear term frequency), then an L2 logistic regression. Two choices are made on folds 1–2: how many past cycles of text a row sees (*n* = 1, 2, 3; P5), and the text model's regularisation (C = 0.1, 0.3, 1).

In [17]:
def tfidf_model(C):
    return make_pipeline(TfidfVectorizer(ngram_range=(1, 2), min_df=10, max_df=0.5, sublinear_tf=True, strip_accents='unicode'),
                         LogisticRegression(C=C, max_iter=3000, random_state=config.SEED))

E1 = {}
for n in [1, 2, 3]:
    for C in [0.1, 0.3, 1]:
        s = text_scores(TEXT[n], lambda: tfidf_model(C), SELECTION_FOLDS)
        E1[(n, C)] = s
        record('E1', f'{n} cycle{"s" if n > 1 else ""} · C={C:g}', stacked_backtest([s], SELECTION_FOLDS), text_only(s, SELECTION_FOLDS))
show('E1')

,setting,PR-AUC (folds 1–2),HR@50 (folds 1–2),text alone: PR-AUC
0,1 cycle · C=0.1,0.4823,33.0%,0.4276
1,1 cycle · C=0.3,0.4827,33.1%,0.4280
2,1 cycle · C=1,0.4824,33.2%,0.4193
3,2 cycles · C=0.1,0.4842,33.1%,0.4447
4,2 cycles · C=0.3,0.4844,33.1%,0.4450
5,2 cycles · C=1,0.4835,33.0%,0.4371
6,3 cycles · C=0.1,0.4810,33.1%,0.4321
7,3 cycles · C=0.3,0.4801,33.0%,0.4322
8,3 cycles · C=1,0.4778,33.1%,0.4267


**Controls: is it really the text?** Two cheaper explanations are tested on the same folds:
- **Violation types (P8):** the text names the kind of problem, which model (b) dropped with the violation-mix group in 03b. Add those 8 counts back instead of the text.
- **Verbosity:** perhaps only *how much* the inspector wrote matters. Add the number of comments and the log of the text length instead of the text.

In [18]:
mix = lr_asof[GROUP_COLS].add_prefix('prev_')
verbosity = pd.DataFrame({'n_comments': WINDOWS[2].comments.map(len), 'log_chars': np.log1p(TEXT[2].str.len())}, index=rows.index)
record('E1 controls', 'model (b) + violation-type counts (8)', backtest(pd.concat([X_b, mix], axis=1), SELECTION_FOLDS))
record('E1 controls', 'model (b) + verbosity (2)', backtest(pd.concat([X_b, verbosity], axis=1), SELECTION_FOLDS))
best_e1 = max([r for r in TRIALS if r['step'] == 'E1'], key=lambda r: r['PR-AUC (folds 1–2)'])['setting']
N_BEST, C_E1 = int(best_e1[0]), float(best_e1.split('C=')[1])
SCORES['E1'] = E1[(N_BEST, C_E1)]
record('E1 controls', f'model (b) + text ({best_e1}), for comparison', stacked_backtest([SCORES['E1']], SELECTION_FOLDS))
show('E1 controls')

,setting,PR-AUC (folds 1–2),HR@50 (folds 1–2)
0,model (b) + violation-type counts (8),0.4774,32.9%
1,model (b) + verbosity (2),0.4779,32.9%
2,"model (b) + text (2 cycles · C=0.3), for comparison",0.4844,33.1%


In [19]:
kept_e1 = decide('E1', [(f'E1 · TF-IDF, {best_e1}', max(r['PR-AUC (folds 1–2)'] for r in TRIALS if r['step'] == 'E1'), ['E1'])])

E1: best configuration "E1 · TF-IDF, 2 cycles · C=0.3" · PR-AUC 0.4844 vs current best (model (b)) 0.4761 → +0.0083 · KEPT: adds value


**Reading E1.**
- **The text carries real information, and it is new.** TF-IDF on its own ranks visits nearly as well as model (b)'s entire structured record (PR-AUC 0.445 vs 0.476 on folds 1–2). Added to model (b), it lifts PR-AUC by +0.008, four times the noise level. Every configuration improves on model (b), and all but one (3 cycles, C = 1) by more than the noise level. **Two cycles of text** is best: one cycle misses useful history, three dilutes it.
- **It is not just violation types or verbosity.** Adding back the 8 violation-type counts gives +0.001, and the two verbosity measures +0.002, both within noise. The words themselves matter.
- **But the headline barely moves.** On folds 1–2 the first-half hit rate rises from 32.9% to 33.1%. PR-AUC weighs the very top of the ranking most, and the gain sits there rather than in which visits make the first half. §8 measures this on the clean year.

**Outcome: E1 is kept** (keep rule met). Current best: model (b) + TF-IDF score (2 cycles, C = 0.3).

## 6 · E2 · Frozen sentence embeddings

A pretrained sentence encoder (`all-MiniLM-L6-v2`, 384 dimensions) embeds each unique comment once, without any training. A row's embedding pools its comments from the chosen window. Two poolings: **mean** (what kinds of things were written) and **sum + log count** (how much of each, as TF-IDF effectively sees it). A logistic regression on the pooled vector gives the text score. Embedding comment by comment means nothing is truncated.

In [20]:
encoder = sentence_transformers.SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2', device=DEVICE)
lists = WINDOWS[N_BEST].comments
unique = pd.unique(pd.Series([c for lst in lists for c in lst]))
t = time.time()
emb = encoder.encode(list(unique), batch_size=256, normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=False)
print(f'{len(unique):,} unique comments embedded in {time.time() - t:.0f} s on {DEVICE}')

position = {c: i for i, c in enumerate(unique)}
n_com = lists.map(len).to_numpy()
mean_vec, sum_vec = np.zeros((len(rows), emb.shape[1]), np.float32), np.zeros((len(rows), emb.shape[1]), np.float32)
for j, lst in enumerate(lists):
    if lst:
        v = emb[[position[c] for c in lst]]
        mean_vec[j], sum_vec[j] = v.mean(0), v.sum(0)
dims = [f'd{i}' for i in range(emb.shape[1])]
POOLED = {'mean': pd.DataFrame(mean_vec, index=rows.index, columns=dims).assign(has_text=(n_com > 0).astype(float)),
          'sum + log count': pd.DataFrame(sum_vec, index=rows.index, columns=dims).assign(log_n=np.log1p(n_com))}

def embedding_model(C):
    return make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=3000, random_state=config.SEED))

E2 = {}
for pooling, inputs in POOLED.items():
    for C in [0.003, 0.03]:
        s = text_scores(inputs, lambda: embedding_model(C), SELECTION_FOLDS)
        E2[(pooling, C)] = s
        record('E2', f'{pooling} · C={C:g}', stacked_backtest([s], SELECTION_FOLDS), text_only(s, SELECTION_FOLDS))
best_e2 = max([r for r in TRIALS if r['step'] == 'E2'], key=lambda r: r['PR-AUC (folds 1–2)'])['setting']
key_e2 = (best_e2.split(' · ')[0], float(best_e2.split('C=')[1]))
SCORES['E2'] = E2[key_e2]
record('E2', f'E1 + E2 ({best_e2}), both scores', stacked_backtest([SCORES[p] for p in BEST['parts']] + [SCORES['E2']], SELECTION_FOLDS))
show('E2')

92,442 unique comments embedded in 23 s on mps


,setting,PR-AUC (folds 1–2),HR@50 (folds 1–2),text alone: PR-AUC
0,mean · C=0.003,0.4779,32.8%,0.3318
1,mean · C=0.03,0.4777,32.8%,0.3274
2,sum + log count · C=0.003,0.4800,32.9%,0.4371
3,sum + log count · C=0.03,0.4790,32.9%,0.4258
4,"E1 + E2 (sum + log count · C=0.003), both scores",0.4837,33.0%,


In [21]:
e2 = {r['setting']: r['PR-AUC (folds 1–2)'] for r in TRIALS if r['step'] == 'E2'}
cands = [(f'E2 · MiniLM, {best_e2}', e2[best_e2], ['E2']),
         (' + '.join(BEST['parts'] + ['E2']), next(v for k, v in e2.items() if 'both' in k), BEST['parts'] + ['E2'])]
r23 = np.corrcoef(logit(SCORES['E1'][2023][year == 2023]), logit(SCORES['E2'][2023][year == 2023]))[0, 1]
print(f'correlation of the E1 and E2 text scores on 2023 rows: {r23:.2f}')
kept_e2 = decide('E2', cands)

correlation of the E1 and E2 text scores on 2023 rows: 0.90
E2: best configuration "E1 + E2" · PR-AUC 0.4837 vs current best (E1 · TF-IDF, 2 cycles · C=0.3) 0.4844 → -0.0007 · adds nothing (within noise)


**Reading E2.** Frozen embeddings carry less usable signal than TF-IDF. Mean pooling is weak (text alone 0.33): it records *what kind* of things were written but not *how much*. Sum pooling recovers most of it (0.437 alone) and adds +0.004 to model (b), half of E1's gain. Adding the embedding score on top of E1 changes nothing (−0.0007). The two text scores correlate at 0.90, so the embeddings find the same information as TF-IDF, only blurrier. A general-purpose encoder places sentences with similar meaning close together, which tends to blur the specific details (what was stored where, at what temperature) that word pairs keep apart.

**Outcome: E2 adds nothing.** This is the first empty step; the stop rule needs two in a row, so E3 runs.

## 7 · E3 · Fine-tuned transformer

A transformer fine-tuned on this task can learn which phrases matter here, which a frozen encoder cannot. It is the most expensive step: each fold year needs 6 fine-tunes (5 out-of-fold groups for the training rows, plus one on all training rows).

**Compute choice (P9, R21).** One epoch on 2,000 rows of 2-cycle text, timed on this Mac:

In [22]:
def finetune_predict(model_name, train_text, train_y, score_text, max_len=256, epochs=2, lr=3e-5, batch_size=32, seed=config.SEED):
    """Fine-tune `model_name` for a hit/no-hit head on train_text; return P(hit) for score_text."""
    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)
    tokenizer = transformers.AutoTokenizer.from_pretrained(model_name)
    model = transformers.AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=1).to(DEVICE)
    encode = lambda texts: {k: v.to(DEVICE) for k, v in tokenizer(list(texts), truncation=True, max_length=max_len, padding=True,
                                                                     return_tensors='pt').items()}
    train_text, score_text = np.asarray(train_text, dtype=object), np.asarray(score_text, dtype=object)
    target = torch.tensor(np.asarray(train_y, float), dtype=torch.float32)
    lengths = np.array([len(t) for t in train_text])
    steps = epochs * int(np.ceil(len(train_text) / batch_size))
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    schedule = transformers.get_linear_schedule_with_warmup(opt, int(0.06 * steps), steps)
    loss_fn = torch.nn.BCEWithLogitsLoss()
    model.train()
    for _ in range(epochs):
        order = np.argsort(lengths + rng.uniform(0, 200, len(lengths)))     # batches of similar length, in random order
        batches = [order[i:i + batch_size] for i in range(0, len(order), batch_size)]
        rng.shuffle(batches)
        for b in batches:
            loss = loss_fn(model(**encode(train_text[b])).logits.squeeze(-1), target[b].to(DEVICE))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); schedule.step(); opt.zero_grad()
    model.eval()
    preds = np.empty(len(score_text))
    order = np.argsort([len(t) for t in score_text])
    with torch.no_grad():
        for i in range(0, len(order), 128):
            b = order[i:i + 128]
            preds[b] = torch.sigmoid(model(**encode(score_text[b])).logits.squeeze(-1).float()).cpu().numpy()
    del model
    if DEVICE == 'mps':
        torch.mps.empty_cache()
    return preds

sample = np.random.default_rng(config.SEED).choice(np.where(year < 2024)[0], 2000, replace=False)
timing = {}
for name in ['distilbert/distilbert-base-uncased', 'answerdotai/ModernBERT-base']:
    t = time.time()
    finetune_predict(name, TEXT[N_BEST].to_numpy()[sample], y[sample], TEXT[N_BEST].to_numpy()[sample[:10]], epochs=1)
    timing[name] = time.time() - t
rows_per_fold = pd.Series({Y: (year < Y).sum() for Y in ALL_FOLDS})
row_epochs = 2 * (rows_per_fold * (1 + 5 * 0.8)).sum()                      # 2 epochs × (all rows + 5 groups at 80%)
pd.DataFrame({'one epoch, 2,000 rows (s)': timing,
              'estimated full E3 run (h)': {k: v / 2000 * row_epochs / 3600 for k, v in timing.items()}}).style.format('{:.1f}')

,"one epoch, 2,000 rows (s)",estimated full E3 run (h)
distilbert/distilbert-base-uncased,19.6,1.3
answerdotai/ModernBERT-base,56.6,3.8


- **Estimated on this Mac (Apple M4 Pro, `mps`):** about 1.3 hours for the whole E3 backtest with DistilBERT, and 3.8 hours with ModernBERT (about three times slower). **Choice: DistilBERT, run locally.** ModernBERT stays an E4 candidate.
- Times printed below are wall-clock and include periods when the machine was idle; keep it awake when re-running (e.g. `caffeinate -i`).

**Fixed settings, not tuned:** 256 tokens (newest comments first; §2.4), 2 epochs, learning rate 3 × 10⁻⁵ with 6% warm-up, batch 32, weight decay 0.01, the project seed. The transformer reads the same 2-cycle text as the selected E1.

The scores take hours, so they are cached in `data/processed/` (git-ignored) under a key naming every setting. A re-run with the same settings loads them, and the cell prints which happened.

In [23]:
E3_SETTINGS = dict(model='distilbert/distilbert-base-uncased', n_cycles=N_BEST, max_len=256, epochs=2, lr=3e-5, batch_size=32, inner=5)
cache = config.PROCESSED / ('nb04_e3_scores_' + '_'.join(str(v).replace('/', '-') for v in E3_SETTINGS.values()) + '.parquet')
if cache.exists():
    e3_table = pd.read_parquet(cache)
    print(f'E3 scores loaded from {cache.name} (delete it to retrain)')
else:
    text = TEXT[N_BEST].to_numpy()
    cols = {}
    t_all = time.time()
    for Y in ALL_FOLDS:
        t = time.time()
        tr, te = np.where(year < Y)[0], np.where(year == Y)[0]
        s = np.full(len(rows), np.nan)
        for a, b in GroupKFold(E3_SETTINGS['inner']).split(tr, groups=lic[tr]):
            s[tr[b]] = finetune_predict(E3_SETTINGS['model'], text[tr[a]], y[tr[a]], text[tr[b]])
        s[te] = finetune_predict(E3_SETTINGS['model'], text[tr], y[tr], text[te])
        cols[str(Y)] = s
        print(f'fold {Y}: {len(tr):,} training rows · {(time.time() - t) / 60:.0f} min')
    e3_table = pd.DataFrame(cols, index=rows.index)
    e3_table.to_parquet(cache)
    print(f'E3 trained in {(time.time() - t_all) / 60:.0f} min; scores cached to {cache.name}')
E3 = {Y: e3_table[str(Y)] for Y in ALL_FOLDS}
SCORES['E3'] = E3
check('R15', 'E3 scores exist for every training row (out-of-fold) and every fold-year row',
      all(E3[Y][year <= Y].notna().all() for Y in ALL_FOLDS))

record('E3', 'DistilBERT (fine-tuned)', stacked_backtest([E3], SELECTION_FOLDS), text_only(E3, SELECTION_FOLDS))
record('E3', ' + '.join(BEST['parts'] + ['E3']) + ', both scores', stacked_backtest([SCORES[p] for p in BEST['parts']] + [E3], SELECTION_FOLDS))
show('E3')

fold 2022: 10,103 training rows · 377 min


fold 2023: 16,384 training rows · 145 min


fold 2024: 21,828 training rows · 39 min
E3 trained in 561 min; scores cached to nb04_e3_scores_distilbert-distilbert-base-uncased_2_256_2_3e-05_32_5.parquet
✓ R15: E3 scores exist for every training row (out-of-fold) and every fold-year row


,setting,PR-AUC (folds 1–2),HR@50 (folds 1–2),text alone: PR-AUC
0,DistilBERT (fine-tuned),0.4803,32.9%,0.4132
1,"E1 + E3, both scores",0.4847,33.0%,


In [24]:
r13 = np.corrcoef(logit(SCORES['E1'][2023][year == 2023]), logit(E3[2023][year == 2023]))[0, 1]
print(f'correlation of the E1 and E3 text scores on 2023 rows: {r13:.2f}')
e3 = {r['setting']: r['PR-AUC (folds 1–2)'] for r in TRIALS if r['step'] == 'E3'}
kept_e3 = decide('E3', [('E3 · DistilBERT', e3['DistilBERT (fine-tuned)'], ['E3']),
                        (' + '.join(BEST['parts'] + ['E3']), next(v for k, v in e3.items() if 'both' in k), BEST['parts'] + ['E3'])])

correlation of the E1 and E3 text scores on 2023 rows: 0.92
E3: best configuration "E1 + E3" · PR-AUC 0.4847 vs current best (E1 · TF-IDF, 2 cycles · C=0.3) 0.4844 → +0.0003 · adds nothing (within noise)


**Reading E3.** Fine-tuning does not beat word counts. The fine-tuned DistilBERT reads the text less well than TF-IDF does (text alone 0.413 vs 0.445 on folds 1–2; below it in every year, §9). Added to model (b) it gives +0.004, and on top of E1 +0.0003. Its score correlates at 0.92 with TF-IDF's: it found the same information, not new information. With about 10,000–22,000 short labelled texts per fold, a 66-million-parameter model has little room to learn more than which words matter, and TF-IDF already captures exactly that.

**Outcome: E3 adds nothing.** That makes two empty steps in a row, so **the stop rule is met and E4 (tuning) is not run.** The settings were not tuned, so this is not proof that no transformer could help. But the gap to TF-IDF (−0.03 PR-AUC on text alone) and the 0.92 correlation make a gain from tuning unlikely, and not worth several more hours of compute.

### Stop rule and E4

In [25]:
ladder = pd.Series({'E1 · TF-IDF': kept_e1, 'E2 · frozen embeddings': kept_e2, 'E3 · fine-tuned DistilBERT': kept_e3}).map(
    {True: 'kept', False: 'adds nothing'})
stop = not kept_e2 and not kept_e3
print(ladder.to_string())
print(f'\nstop rule (two consecutive steps add nothing): {"MET, so E4 (tuning) is not run" if stop else "not met, so E4 tuning runs"}')
print(f'final text model: {BEST["name"]} · folds 1–2 PR-AUC {BEST["ap"]:.4f} (model (b): {base_ap:.4f})')

E1 · TF-IDF                           kept
E2 · frozen embeddings        adds nothing
E3 · fine-tuned DistilBERT    adds nothing

stop rule (two consecutive steps add nothing): MET, so E4 (tuning) is not run
final text model: E1 · TF-IDF, 2 cycles · C=0.3 · folds 1–2 PR-AUC 0.4844 (model (b): 0.4761)


## 8 · The clean year: does text move the headline?

The selected text model now runs through the full backtest. **2024 is the clean year:** no text setting was chosen on it. It is compared with model (b) (does text add anything?) and Rule 1 (the +3 bar).

In [26]:
FINAL_NAME = BEST['name']
# E1 and E2 were scored on folds 1-2 only; rerun them on all three fold years with the selected settings (E3 already has 2024)
FULL = {'E1': text_scores(TEXT[N_BEST], lambda: tfidf_model(C_E1), ALL_FOLDS),
        'E2': text_scores(POOLED[key_e2[0]], lambda: embedding_model(key_e2[1]), ALL_FOLDS), 'E3': E3}
check('P2', 'the selection-fold scores are reproduced when extended to 2024',
      all(np.allclose(FULL[p][Y], SCORES[p][Y], equal_nan=True) for p in ['E1', 'E2'] for Y in SELECTION_FOLDS))
print(f'final text model: {FINAL_NAME} (stacks {", ".join(BEST["parts"])})')
oof_text = stacked_backtest([FULL[p] for p in BEST['parts']], ALL_FOLDS)
ev['text_model'] = oof_text.loc[ev.index].to_numpy()

METHODS = [('actual', 'Current practice'), ('rule1', 'Rule 1 · previous inspection'), ('model_b', 'Model (b)'),
           ('text_model', f'Model (b) + text ({FINAL_NAME})'), ('oracle', 'Oracle')]

def compare(e, label):
    pm = {k: per_month(e, k) for k, _ in METHODS}
    base, orc, r1, mb = (hit_rate(pm[k]) for k in ['actual', 'oracle', 'rule1', 'model_b'])
    out = {}
    for k, name in METHODS:
        hr = hit_rate(pm[k])
        lo1, hi1 = bootstrap_diff(pm[k], pm['rule1'])
        lob, hib = bootstrap_diff(pm[k], pm['model_b'])
        out[name] = {f'HR@50 ({label})': hr,
                     'vs Rule 1 (per 100)': 100 * (hr - r1), 'vs Rule 1 95% CI': '—' if k == 'rule1' else f'[{100 * lo1:+.1f}, {100 * hi1:+.1f}]',
                     'vs model (b) (per 100)': 100 * (hr - mb), 'vs model (b) 95% CI': '—' if k == 'model_b' else f'[{100 * lob:+.1f}, {100 * hib:+.1f}]',
                     'headroom captured': (hr - base) / (orc - base), 'days earlier': days_earlier(e, k),
                     'PR-AUC': average_precision_score(e.hit, e[k])}
    return pd.DataFrame(out).T

FMT = {'vs Rule 1 (per 100)': '{:+.1f}', 'vs model (b) (per 100)': '{:+.1f}', 'headroom captured': '{:.0%}', 'days earlier': '{:.1f}', 'PR-AUC': '{:.3f}'}
clean = compare(ev[ev.visit_ts.dt.year == 2024], '2024, clean')
clean.style.format(FMT | {'HR@50 (2024, clean)': '{:.1%}'})

✓ P2: the selection-fold scores are reproduced when extended to 2024
final text model: E1 · TF-IDF, 2 cycles · C=0.3 (stacks E1)


,"HR@50 (2024, clean)",vs Rule 1 (per 100),vs Rule 1 95% CI,vs model (b) (per 100),vs model (b) 95% CI,headroom captured,days earlier,PR-AUC
Current practice,23.8%,-8.5,"[-10.6, -6.6]",-11.0,"[-12.5, -9.5]",0%,0.0,0.245
Rule 1 · previous inspection,32.4%,+0.0,—,-2.4,"[-3.5, -1.5]",38%,3.4,0.358
Model (b),34.8%,+2.4,"[+1.5, +3.5]",+0.0,—,48%,4.9,0.423
"Model (b) + text (E1 · TF-IDF, 2 cycles · C=0.3)",34.8%,+2.5,"[+1.6, +3.6]",+0.0,"[-0.3, +0.4]",49%,4.9,0.432
Oracle,46.5%,+14.1,"[+12.1, +16.2]",+11.7,"[+9.8, +13.6]",100%,10.9,1.000


In [27]:
by_year = []
for Y in ALL_FOLDS:
    e = ev[ev.visit_ts.dt.year == Y]
    pm = {k: per_month(e, k) for k in ['rule1', 'model_b', 'text_model']}
    lo, hi = bootstrap_diff(pm['text_model'], pm['model_b'])
    by_year.append({'year': Y, 'role': 'clean check' if Y == 2024 else 'selection',
                    'PR-AUC model (b)': average_precision_score(e.hit, e.model_b), 'PR-AUC + text': average_precision_score(e.hit, e.text_model),
                    'text vs model (b), per 100': 100 * (hit_rate(pm['text_model']) - hit_rate(pm['model_b'])), '95% CI': f'[{100 * lo:+.1f}, {100 * hi:+.1f}]',
                    'text vs Rule 1, per 100': 100 * (hit_rate(pm['text_model']) - hit_rate(pm['rule1']))})
by_year = pd.DataFrame(by_year).set_index('year')
by_year.style.format({'PR-AUC model (b)': '{:.3f}', 'PR-AUC + text': '{:.3f}', 'text vs model (b), per 100': '{:+.1f}', 'text vs Rule 1, per 100': '{:+.1f}'})

,role,PR-AUC model (b),PR-AUC + text,"text vs model (b), per 100",95% CI,"text vs Rule 1, per 100"
year,,,,,,
2022,selection,0.466,0.473,+0.1,"[-0.1, +0.4]",+3.1
2023,selection,0.486,0.496,+0.2,"[-0.2, +0.5]",+2.5
2024,clean check,0.423,0.432,+0.0,"[-0.3, +0.4]",+2.5


**Where in the list does the text help?** The capture curve answers it: the share of each month's hits found after the first 10%, 25% and 50% of the month's visits, for model (b) with and without text (2024).

In [28]:
e24 = ev[ev.visit_ts.dt.year == 2024]
cuts = [0.05, 0.10, 0.25, 0.50]
cap = pd.DataFrame({name: [per_month(e24, k, q).hits.sum() / per_month(e24, k, q).month_hits.sum() for q in cuts]
                    for k, name in [('rule1', 'Rule 1'), ('model_b', 'Model (b)'), ('text_model', 'Model (b) + text')]},
                   index=[f'first {q:.0%} of visits' for q in cuts])
cap['text − model (b), points'] = 100 * (cap['Model (b) + text'] - cap['Model (b)'])
brier = {k: brier_score_loss(e24.hit, e24[k]) for k in ['model_b', 'text_model']}
print(f'calibration, 2024 Brier score: model (b) {brier["model_b"]:.4f} · with text {brier["text_model"]:.4f} '
      f'(constant: {brier_score_loss(e24.hit, np.full(len(e24), e24.hit.mean())):.4f})')
cap.style.format({c: '{:.1%}' for c in cap.columns[:3]} | {'text − model (b), points': '{:+.1f}'})

calibration, 2024 Brier score: model (b) 0.1608 · with text 0.1597 (constant: 0.1783)


,Rule 1,Model (b),Model (b) + text,"text − model (b), points"
first 5% of visits,9.2%,11.3%,11.9%,+0.5
first 10% of visits,19.0%,21.5%,22.6%,+1.1
first 25% of visits,41.6%,47.3%,47.4%,+0.2
first 50% of visits,69.6%,74.8%,74.9%,+0.1


**Which visits does the text move?** Within each 2024 month: the visits the text model puts in the top 10% that model (b) ranked lower, and those it pushes out of the top 10%.

In [29]:
pct = {k: e24.groupby('month')[k].rank(ascending=False, pct=True) for k in ['model_b', 'text_model']}
top_b, top_t = pct['model_b'] <= 0.10, pct['text_model'] <= 0.10
moved = pd.DataFrame({name: {'visits': m.sum(), 'hit rate': e24.hit[m].mean(), 'model (b) score (mean)': e24.model_b[m].mean()}
                      for name, m in [('moved into the top 10% by the text', top_t & ~top_b),
                                      ('pushed out of the top 10% by the text', top_b & ~top_t),
                                      ('in the top 10% either way', top_b & top_t)]}).T
moved.style.format({'visits': '{:,.0f}', 'hit rate': '{:.0%}', 'model (b) score (mean)': '{:.0%}'})

,visits,hit rate,model (b) score (mean)
moved into the top 10% by the text,71,54%,39%
pushed out of the top 10% by the text,71,32%,46%
in the top 10% either way,487,53%,58%


**Reading the clean year (2024).**
- **The headline does not move.** Model (b) + text finds Priority violations in 34.8% of first-half visits, the same as model (b) (+0.0 per 100 visits, 95% CI −0.3 to +0.4). Against Rule 1 it is +2.5 [+1.6, +3.6]: **the +3 bar is still not met.** The same holds in 2022 (+0.1) and 2023 (+0.2).
- **The ranking does improve, consistently.** PR-AUC rises in every year (+0.007, +0.010, +0.009; 2024: 0.423 → 0.432), and calibration improves slightly (Brier 0.1608 → 0.1597).
- **The gain is at the very top of the list.** The share of the month's hits found in the first 10% of visits rises from 21.5% to 22.6% (+1.1 points), and in the first 5% by +0.5. By a quarter of the month the difference has gone. The 71 visits the text moves into a month's top 10% find a Priority violation 54% of the time; the 71 it pushes out, 32%.
- **What that is worth:** with about 470 visits and 110 hits a month, the top 10% is the first two to three working days. The text finds about one more hit a month there; by mid-month both orders have found the same hits. If the city plans by the month or the half-month, the text adds nothing it would notice. If it acts on the very top of the list (say, the first few days, or a short list of urgent visits), the text helps a little.

## 9 · What the text model reads, and whether it reads the kitchen or the inspector (P6, R17)

Three checks on the selected TF-IDF text model, fit on 2020–2023:
1. **Which words move the score.** Any term shown must pass the name scan (P7).
2. **Stability by year:** the text-only score's PR-AUC in each fold year. A model that learned 2020-era writing habits would fade.
3. **Kitchen or inspector?** Inspector identity is not public, so this cannot be settled. Two indirect signs: the text score's correlation with local strictness (model (b)'s area rates, the best available inspector proxy), and whether the text's gain is concentrated where local strictness is high.

In [30]:
lower_counts = Counter(w for t in com.text.drop_duplicates() for w in re.findall(r'\b[a-z]+\b', t))
CALENDAR = {'january', 'february', 'march', 'april', 'may', 'june', 'july', 'august', 'september', 'october', 'november',
            'december', 'jan', 'feb', 'mar', 'apr', 'jun', 'jul', 'aug', 'sep', 'sept', 'oct', 'nov', 'dec', 'monday',
            'tuesday', 'wednesday', 'thursday', 'friday', 'saturday', 'sunday'}

def possibly_a_name(word):
    """A capitalised word that is rarely written in lower case anywhere in the comments (fewer than 20 times)."""
    return word[:1].isupper() and word.lower() not in CALENDAR and lower_counts[word.lower()] < 20

def safe_snippet(text, width=160):
    """Mask likely names, then cut to `width` characters. Every comment shown in this notebook goes through this."""
    out = re.sub(r'\b[A-Z][a-z]+\b', lambda m: '[…]' if possibly_a_name(m.group()) else m.group(), clean_comment(text))
    return out if len(out) <= width else out[:width].rsplit(' ', 1)[0] + ' …'

def scan_ok(shown):
    """P7 display check: no email, phone, title + name, or likely name left in anything displayed."""
    return all(not (EMAIL.search(s) or PHONE.search(s) or TITLED.search(s)
                    or any(possibly_a_name(w) for w in re.findall(r'\b[A-Z][a-z]+\b', s))) for s in shown)

tm = tfidf_model(C_E1).fit(TEXT[N_BEST][year < 2024], y[year < 2024])
vocab, coef = tm[0].get_feature_names_out(), tm[1].coef_[0]
order = np.argsort(coef)
terms = pd.DataFrame({'raise the score': vocab[order[-20:]][::-1], 'lower the score': vocab[order[:20]]},
                     index=pd.RangeIndex(1, 21, name='rank'))
check('R18', 'no likely names among the displayed terms',
      not any(lower_counts[w] < 20 and not w.isdigit() for t in terms.to_numpy().ravel() for w in t.split()))
terms

✓ R18: no likely names among the displayed terms


,raise the score,lower the score
rank,,
1,chicken,service
2,multiple,the service
3,prep,the kitchen
4,discontinue,exhausthood sticker
5,removed,or replace
6,cos,clean floor
7,containers,sandwiches
8,from,service area
9,raw,coffee


In [31]:
e1_full = FULL['E1']
stab = pd.DataFrame({Y: {'TF-IDF text alone: PR-AUC': average_precision_score(y[year == Y], e1_full[Y][year == Y]),
                         'DistilBERT text alone: PR-AUC': average_precision_score(y[year == Y], E3[Y][year == Y]),
                         'model (b): PR-AUC': average_precision_score(y[year == Y], oof_b[year == Y]),
                         'hit rate': y[year == Y].mean()} for Y in ALL_FOLDS}).T
stab.style.format('{:.3f}')

,TF-IDF text alone: PR-AUC,DistilBERT text alone: PR-AUC,model (b): PR-AUC,hit rate
2022,0.439,0.416,0.466,0.221
2023,0.451,0.411,0.486,0.192
2024,0.418,0.404,0.423,0.232


In [32]:
diag = ev.assign(text_score=logit(text_only(e1_full, ALL_FOLDS).loc[ev.index]),
                 strictness=X_b.loc[ev.index, 'area_severe_rate_180d'].to_numpy(),
                 has_text=(TEXT[N_BEST].loc[ev.index].str.len() > 0).to_numpy())
r_strict = diag[['text_score', 'strictness']].corr().iloc[0, 1]
r_record = np.corrcoef(diag.text_score, logit(diag.model_b))[0, 1]
print(f'correlation of the text score with local strictness (area severe rate, 180 days): {r_strict:.2f}; '
      f'with model (b)\'s score: {r_record:.2f}')

diag['strictness band'] = pd.qcut(diag.strictness, 3, labels=['least strict third', 'middle third', 'strictest third'])
groups = {'all': diag}
groups |= {f'license type {c}': diag[diag.category == c] for c in ['FS', 'FT', 'RF']}
groups |= {str(b): g for b, g in diag.groupby('strictness band', observed=True)}
groups |= {'with earlier comments': diag[diag.has_text], 'no earlier comments': diag[~diag.has_text]}
sg = {}
for name, g in groups.items():
    sg[name] = {'visits (2022–24)': len(g), 'PR-AUC model (b)': average_precision_score(g.hit, g.model_b),
                'PR-AUC + text': average_precision_score(g.hit, g.text_model)}
sg = pd.DataFrame(sg).T
sg['gain'] = sg['PR-AUC + text'] - sg['PR-AUC model (b)']
sg.style.format({'visits (2022–24)': '{:,.0f}', 'PR-AUC model (b)': '{:.3f}', 'PR-AUC + text': '{:.3f}', 'gain': '{:+.3f}'})

correlation of the text score with local strictness (area severe rate, 180 days): 0.26; with model (b)'s score: 0.81


,visits (2022–24),PR-AUC model (b),PR-AUC + text,gain
all,"17,354",0.454,0.462,+0.009
license type FS,"7,843",0.466,0.472,+0.006
license type FT,"7,065",0.458,0.471,+0.012
license type RF,"2,446",0.381,0.380,-0.001
least strict third,"5,386",0.282,0.287,+0.006
middle third,"5,383",0.440,0.453,+0.013
strictest third,"5,385",0.522,0.532,+0.010
with earlier comments,"13,070",0.474,0.484,+0.010
no earlier comments,"4,284",0.171,0.172,+0.001


**Reading §9.**
- **The text reads the kitchen.** The words that raise the score describe kitchens that handle raw proteins and do a lot of prep: *chicken, raw, rice, beef, prep, cookline, containers, stored*, plus *discontinue* and *cos* (corrected on site), which mark practices the inspector had to stop. The words that lower it describe low-risk menus (*coffee, sandwiches*) and maintenance items (*repair or replace, tiles, floor area, exhaust hood sticker*). This is information model (b) lacks: it knows *how many* violations a place had, not *what kind of cooking* goes on there.
- **Some of it may be the inspector.** Phrases like *the service, expired service, exhausthood sticker* read like particular inspectors' habits for recording hood-service paperwork. They point toward lower risk, which fits an inspector who records minor paperwork carefully. The text score correlates only 0.26 with local strictness (vs 0.81 with model (b)'s score), and its gain is not concentrated in the strictest areas (+0.006, +0.013, +0.010 by third). There is no sign that the text mainly identifies inspectors, but without inspector IDs this cannot be ruled out (R17).
- **Stable over time:** TF-IDF on text alone scores 0.44, 0.45, 0.42 in 2022–2024, tracking model (b) (0.47, 0.49, 0.42). It does not fade as writing styles drift. DistilBERT is below TF-IDF in every year.
- **Where it helps:** take-out restaurants (+0.012) and sit-down restaurants (+0.006), not retail stores (−0.001). It helps only rows with earlier comments, as expected: about a quarter of visits have none (first visits, or clean earlier visits).

## 10 · Comment snippets as explanations (P10)

Even if accuracy gains are small, a sentence from the last inspection can tell an inspector *why* a place is near the top of the list. For the TF-IDF model, each comment's contribution is the sum of its terms' weights. Shown below: the five highest-ranked 2024 visits in one month, with the comment that pushed each score up the most. Every snippet passes the name scan.

In [33]:
def top_comment(comments):
    if not comments:
        return '', 0.0
    contrib = tm[0].transform(comments) @ coef
    i = int(np.argmax(contrib))
    return comments[i], float(contrib[i])

month = e24.month.value_counts().index[0]
top = e24[e24.month == month].sort_values('text_model', ascending=False).head(5)
expl = []
for idx, r in top.iterrows():
    comment, c = top_comment(WINDOWS[N_BEST].comments.loc[idx])
    expl.append({'risk score': r.text_model, 'model (b) score': r.model_b, 'Priority violation found': 'yes' if r.hit else 'no',
                 'comment that raised the score most': safe_snippet(comment) if comment else '(no earlier comments)'})
expl = pd.DataFrame(expl, index=pd.RangeIndex(1, len(expl) + 1, name=f'rank in {month}'))
check('R18', 'displayed snippets pass the name scan', scan_ok(expl['comment that raised the score most']))
expl.style.format({'risk score': '{:.0%}', 'model (b) score': '{:.0%}'})

✓ R18: displayed snippets pass the name scan


,risk score,model (b) score,Priority violation found,comment that raised the score most
rank in 2024-10,,,,
1,79%,69%,yes,Raw Chicken stored on top of raw pork and raw pork stored on top of raw chicken- Discontinue storing raw meat products together to prevent cross-contamination. …
2,78%,74%,no,Clean interior / exterior of all house refrigerator unit of food soils. Grease build up in interior fryer cabinet. Fryer basket with crusted food sides of …
3,76%,74%,no,Observed shaved steak / raw hamburgers / pizza sauce stored in low boy reach in unit between 48F-52F. Discussed with PIC to maintain cold holding @ 41F or …
4,76%,78%,yes,Clean inside / outside of all refrigeration / freezer units - all cooking equipment - under sall prep tables - exterior of the bulk food bins-from visible food …
5,75%,77%,yes,Hands should be clean at the start of processing any foods - Train staff the importance of washing hands- Corrected on site Staff washed hands at appropiate …


**Reading §10.** As explanations, the snippets are mixed. Rank 1 (raw meats stored together), rank 3 (food held at 48–52 °F) and rank 5 (handwashing) are specific, food-safety-relevant reasons an inspector would recognise. Ranks 2 and 4 are generic cleaning instructions: the comment *pushed the score up* because of words like "food", "prep" and "cooking", but it is not a reason to visit first. The top places are also already near the top under model (b) (scores 69–78%), so the snippet adds context rather than changing the decision.

**Implication (P10):** if the list is ever used, show inspectors the **most recent inspection's comments** next to each place, which needs no model, rather than presenting model-picked snippets as "the reason". Model-picked snippets look authoritative but are only partly meaningful.

## 11 · Findings

**Results**

| | Model (b) | Model (b) + TF-IDF text |
|---|---|---|
| Folds 1–2 (selection): PR-AUC | 0.476 | 0.484 |
| **2024 (clean): first-half hit rate** | **34.8%** | **34.8%** |
| 2024 vs model (b), per 100 visits | — | +0.0 [−0.3, +0.4] |
| 2024 vs Rule 1, per 100 visits | +2.4 [+1.5, +3.5] | +2.5 [+1.6, +3.6] |
| 2024: PR-AUC · share of hits in the first 10% of visits | 0.423 · 21.5% | 0.432 · 22.6% |

| Step | Text alone, PR-AUC (folds 1–2) | Added to the current best | Outcome |
|---|---|---|---|
| E1 · TF-IDF (2 cycles) | 0.445 | +0.008 over model (b) | **kept** |
| E2 · frozen MiniLM embeddings | 0.437 | −0.001 over E1 | adds nothing |
| E3 · fine-tuned DistilBERT | 0.413 | +0.000 over E1 | adds nothing → **stop** |

1. **Inspector comments contain real information that the structured record lacks**, mainly *what kind of kitchen* a place is (raw proteins, heavy prep) rather than *how many* violations it had. The PR-AUC gain is consistent in all three years and is not explained by violation types or by how much the inspector wrote.
2. **It does not change the headline.** The first-half hit rate is unchanged on the clean year (+0.0 per 100 visits), and the +3 bar over Rule 1 is still not met (+2.5). The gain is concentrated in the first 10% of the list: about one extra hit a month.
3. **The simplest text model is the best.** TF-IDF beats frozen embeddings, which beat a fine-tuned transformer; the more complex models found the same information, only less sharply. This repeats Phase 1's lesson: **the limit is the information, not the algorithm.** The stop rule ended the ladder after E3; E4 was not run.
4. **The pre-check (P3) holds as a raw association, but most of it overlaps with model (b):** more comments, and mentions of rodents, cockroaches or "repeat", mark riskier places, and model (b) already expects most of that risk.
5. **Checks passed:** text strictly before the decision date, plus a truncation test with 0 mismatches (R15); no truncation for TF-IDF or the embeddings, and the newest comments kept for DistilBERT (R16); stable by year, with no sign of an inspector proxy, though that can't be ruled out (R17); personal data masked before modelling, and every displayed term and snippet scanned (R18).

**Carried to notebook 05:** model (b) + TF-IDF text as the primary model, with model (b) as the pre-declared secondary. Both are retrained on 2020–2024 and scored once on the test period.

## 12 · Experiment log

In [34]:
log = pd.DataFrame(TRIALS).rename(columns={'setting': 'experiment', 'PR-AUC (folds 1–2)': 'PR-AUC', 'HR@50 (folds 1–2)': 'HR@50'})
log['note'] = 'selection score on folds 1–2 (2022–23); step ' + log.pop('step')
log['params'] = log.pop('text alone: PR-AUC').map(lambda v: '' if pd.isna(v) else f'text alone PR-AUC {v:.4f}')
final = clean.reset_index(names='experiment').rename(columns={'HR@50 (2024, clean)': 'HR@50'})
final = final[final.experiment.str.startswith('Model (b)')]
final['note'] = 'final check on 2024 (clean)'
final['params'] = final.experiment.map({f'Model (b) + text ({FINAL_NAME})': str(dict(text=BEST['parts'], n_cycles=N_BEST, tfidf_C=C_E1,
                                                                                    e2=key_e2, e3=E3_SETTINGS))}).fillna('')
log = pd.concat([log, final[['experiment', 'HR@50', 'vs Rule 1 (per 100)', 'vs Rule 1 95% CI', 'days earlier', 'PR-AUC', 'note', 'params']]],
                ignore_index=True)
log.insert(0, 'date', date.today().isoformat())
log.insert(1, 'notebook', '04_text_models')
path = config.RESULTS / 'experiments.csv'
previous = pd.read_csv(path)
previous = previous[previous.notebook != '04_text_models']       # re-running replaces this notebook's rows
pd.concat([previous, log], ignore_index=True).to_csv(path, index=False)
print(f'wrote {len(log)} rows for this notebook to {path.relative_to(config.ROOT)}')
pd.DataFrame(CHECKS)

wrote 22 rows for this notebook to results/experiments.csv


,risk,check,result,detail
0,R14,no test-period rows built,pass,"27,457 rows"
1,P1,model (b) has its 33 features,pass,"27,457 rows × 33 features"
2,R23,model (b) reproduced exactly (2024 first-half hit rate),pass,0.3480 here vs 0.3480 in notebook 03b
3,R18,no email addresses or phone numbers left in the cleaned text,pass,"812,396 comments"
4,R4,comments carry almost no dates from after their visit (< 0.1% of all comments),pass,"47 of 812,396"
5,R15,1-cycle text comes only from visits before t0,pass,65% of rows have text
6,R15,2-cycle text comes only from visits before t0,pass,77% of rows have text
7,R15,3-cycle text comes only from visits before t0,pass,81% of rows have text
8,R15,text unchanged when all later data is deleted (truncation test),pass,"19,263 rows, 0 mismatches"
9,R15,E3 scores exist for every training row (out-of-fold) and every fold-year row,pass,
